# Anthropic Economic Index — Análise do Brasil

Exploração do dataset `Anthropic/EconomicIndex` com foco no uso de IA no Brasil vs. países de perfil similar.

**Dataset:** [Anthropic/EconomicIndex](https://huggingface.co/datasets/Anthropic/EconomicIndex)  
**Release:** `release_2025_09_15` — primeira com dados geográficos por país  
**Data:** Abril 2026

---
### Schema — `aei_enriched_claude_ai_*.csv` (formato long)
| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `geo_id` | str | ISO-3 do país (`BRA`, `USA`, …) ou código de estado dos EUA |
| `geography` | str | `country` \| `state_us` \| `global` |
| `facet` | str | Dimensão: `country`, `onet_task`, `collaboration`, `request` |
| `variable` | str | Métrica: `usage_per_capita_index`, `onet_task_pct`, `automation_pct`, … |
| `cluster_name` | str | Entidade dentro do facet (nome da tarefa, padrão de colaboração, etc.) |
| `value` | float | Valor numérico |

## 0. Instalação de dependências

In [ ]:
%pip install -q huggingface_hub pandas matplotlib seaborn plotly openpyxl

## 1. Setup e Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../outputs')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

BRAZIL      = 'BRA'
COMPARABLES = ['BRA', 'ARG', 'MEX', 'IND', 'ZAF', 'COL', 'CHL']
LABELS      = {
    'BRA': 'Brasil',       'ARG': 'Argentina',    'MEX': 'México',
    'IND': 'Índia',        'ZAF': 'África do Sul', 'COL': 'Colômbia',
    'CHL': 'Chile'
}

print('Setup concluído.')

## 2. Download do Dataset

> Os caminhos abaixo são fixos — o arquivo principal está em Git LFS e não aparece no `list_repo_files` padrão.

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID   = 'Anthropic/EconomicIndex'
REPO_TYPE = 'dataset'

# Arquivos conhecidos no repositório
FILES = {
    'aei_enriched': 'release_2025_09_15/data/output/aei_enriched_claude_ai_2025-08-04_to_2025-08-11.csv',
    'pop_country':  'release_2025_09_15/data/input/working_age_pop_2024_country_raw.csv',
    'soc_struct':   'release_2025_09_15/data/input/soc_structure_raw.csv',
    'task_pct':     'release_2025_09_15/data/input/task_pct_v2.csv',
    'job_exposure': 'labor_market_impacts/job_exposure.csv',
    'task_penetr':  'labor_market_impacts/task_penetration.csv',
}

def download(key):
    path = hf_hub_download(
        repo_id=REPO_ID,
        filename=FILES[key],
        repo_type=REPO_TYPE,
        local_dir=DATA_DIR
    )
    print(f'✓ {key}: {path}')
    return Path(path)

paths = {k: download(k) for k in FILES}
print('\nDownload concluído.')

## 3. Exploração Inicial

In [ ]:
df = pd.read_csv(paths['aei_enriched'])

print(f'Shape: {df.shape}')
print(f'Colunas: {df.columns.tolist()}')
df.head(3)

In [ ]:
# Valores únicos das dimensões de análise
for col in ['geography', 'facet', 'variable']:
    vals = df[col].value_counts()
    print(f'\n--- {col} ({len(vals)} únicos) ---')
    print(vals.to_string())

In [ ]:
countries = df[df['geography'] == 'country']['geo_id'].unique()
print(f'Países no dataset: {len(countries)}')
print(f'Brasil (BRA) presente: {BRAZIL in countries}')
print(f"Período: {df['date_start'].min()} → {df['date_end'].max()}")

In [ ]:
# Auxiliares
df_pop = pd.read_csv(paths['pop_country'])
df_soc = pd.read_csv(paths['soc_struct'])
df_job = pd.read_csv(paths['job_exposure'])   # occ_code, title, observed_exposure

print('Population:', df_pop.shape, df_pop.columns.tolist())
print('SOC struct:', df_soc.shape, df_soc.columns.tolist())
print('Job exposure:', df_job.shape, df_job.columns.tolist())
df_job.head()

## 4. Brasil vs. Países Comparáveis

### 4.1 Uso per capita indexado
> `usage_per_capita_index` — razão entre uso per capita do país e a média global. > 1 = acima da média.

In [ ]:
df_usage = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable']  == 'usage_per_capita_index') &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(country=lambda x: x['geo_id'].map(LABELS))
    .sort_values('value', ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#1565C0' if g == BRAZIL else '#90CAF9' for g in df_usage['geo_id']]
ax.barh(df_usage['country'], df_usage['value'], color=colors)
ax.axvline(1, color='gray', linestyle='--', linewidth=1.2, label='Média global')
ax.set_xlabel('Índice de uso per capita (1 = média global)')
ax.set_title('Adoção de IA — uso per capita indexado')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'usage_per_capita_index.png', dpi=150)
plt.show()

print(df_usage[['country', 'value']].rename(columns={'value': 'index'}).to_string(index=False))

### 4.2 Automação vs. Augmentação
> `automation_pct` = modelo executa sem supervisão contínua.  
> `augmentation_pct` = humano mantém controle e validação.

In [ ]:
df_auto = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable'].isin(['automation_pct', 'augmentation_pct'])) &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(
        country=lambda x: x['geo_id'].map(LABELS),
        variable=lambda x: x['variable'].str.replace('_pct', '')
    )
    .pivot_table(index=['country', 'geo_id'], columns='variable', values='value')
    .reset_index()
    .sort_values('automation', ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(df_auto))
w = 0.35
ax.bar([i - w/2 for i in x], df_auto['automation'],  width=w, label='Automação',   color='#EF5350')
ax.bar([i + w/2 for i in x], df_auto['augmentation'], width=w, label='Augmentação', color='#42A5F5')
ax.set_xticks(list(x))
ax.set_xticklabels(df_auto['country'], rotation=20, ha='right')
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Automação vs. Augmentação por País')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'automation_vs_augmentation.png', dpi=150)
plt.show()

### 4.3 Top tarefas O\*NET no Brasil

In [ ]:
df_tasks_br = (
    df[
        (df['geo_id']   == BRAZIL) &
        (df['facet']    == 'onet_task') &
        (df['variable'] == 'onet_task_pct')
    ]
    .nlargest(15, 'value')
    .sort_values('value')
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_tasks_br['cluster_name'], df_tasks_br['value'], color='#1565C0')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title('Top 15 Tarefas O*NET — Brasil', fontsize=13)
ax.set_xlabel('% do uso total no país')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'brazil_top_onet_tasks.png', dpi=150)
plt.show()

### 4.4 Heatmap de especialização — tarefa × país
> `onet_task_pct_index` > 1: país sobrerrepresentado nessa tarefa vs. média global.

In [ ]:
df_idx = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'onet_task') &
        (df['variable']  == 'onet_task_pct_index') &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .assign(country=lambda x: x['geo_id'].map(LABELS))
)

# Top 10 tarefas mais diferenciadas no Brasil
top_tasks = (
    df_idx[df_idx['geo_id'] == BRAZIL]
    .nlargest(10, 'value')['cluster_name'].tolist()
)

df_heat = (
    df_idx[df_idx['cluster_name'].isin(top_tasks)]
    .pivot_table(index='cluster_name', columns='country', values='value')
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(df_heat, annot=True, fmt='.2f', cmap='RdYlGn',
            center=1, linewidths=0.4, ax=ax)
ax.set_title('Índice de especialização por tarefa O*NET (1 = média global)', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'heatmap_task_specialization.png', dpi=150)
plt.show()

## 5. Visualizações Interativas (Plotly)

### 5.1 Padrões de colaboração no Brasil

In [ ]:
df_collab = (
    df[
        (df['geo_id']   == BRAZIL) &
        (df['facet']    == 'collaboration') &
        (df['variable'] == 'collaboration_pct_index')
    ]
    .sort_values('value', ascending=False)
)

fig = px.bar(
    df_collab, x='cluster_name', y='value',
    title='Brasil — Especialização por padrão de colaboração<br><sup>Índice relativo à média global (1 = média)</sup>',
    labels={'cluster_name': 'Padrão', 'value': 'Índice'},
    color='value', color_continuous_scale='RdYlGn', color_continuous_midpoint=1
)
fig.add_hline(y=1, line_dash='dash', line_color='gray')
fig.update_layout(coloraxis_showscale=False, xaxis_title='')
fig.write_html(OUTPUT_DIR / 'brazil_collaboration_index.html')
fig.show()

### 5.2 Scatter: adoção × automação com tamanho por população

In [ ]:
df_scatter = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable'].isin(['usage_per_capita_index', 'automation_pct'])) &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .pivot_table(index='geo_id', columns='variable', values='value')
    .reset_index()
    .assign(country=lambda x: x['geo_id'].map(LABELS))
)

# Enriquecer com população
pop_col = [c for c in df_pop.columns if 'working_age' in c.lower() or 'pop' in c.lower()][-1]
iso_col = [c for c in df_pop.columns if 'iso' in c.lower() or 'alpha' in c.lower()][0]
df_scatter = df_scatter.merge(
    df_pop[[iso_col, pop_col]].rename(columns={iso_col: 'geo_id', pop_col: 'pop'}),
    on='geo_id', how='left'
)

fig = px.scatter(
    df_scatter,
    x='usage_per_capita_index', y='automation_pct',
    text='country', size='pop', size_max=60,
    title='Adoção vs. Automação — países comparáveis<br><sup>Tamanho = população em idade ativa</sup>',
    labels={'usage_per_capita_index': 'Índice de uso per capita',
            'automation_pct': 'Taxa de automação'},
    color='country', color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='top center')
fig.add_vline(x=1, line_dash='dot', line_color='lightgray')
fig.update_layout(showlegend=False, yaxis_tickformat='.0%')
fig.write_html(OUTPUT_DIR / 'scatter_adoption_vs_automation.html')
fig.show()

### 5.3 Treemap — Top 30 tarefas O\*NET no Brasil

In [ ]:
df_tree = (
    df[
        (df['geo_id']   == BRAZIL) &
        (df['facet']    == 'onet_task') &
        (df['variable'] == 'onet_task_pct')
    ]
    .nlargest(30, 'value')
    .assign(
        group=lambda x: x['cluster_name'].str.split('::').str[0],
        task =lambda x: x['cluster_name'].str.split('::').str[-1]
    )
)

fig = px.treemap(
    df_tree, path=['group', 'task'], values='value',
    title='Brasil — Top 30 Tarefas O*NET por volume de uso',
    color='value', color_continuous_scale='Blues',
    hover_data={'value': ':.2%'}
)
fig.update_traces(textinfo='label+percent entry')
fig.write_html(OUTPUT_DIR / 'brazil_treemap_onet_tasks.html')
fig.show()

## 6. Exposição por ocupação (labor_market_impacts)

> `job_exposure.csv` — exposição observada de cada ocupação (SOC code) ao uso real do Claude.

In [ ]:
df_job_sorted = df_job.sort_values('observed_exposure', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 15 mais expostas
top15 = df_job_sorted.head(15).sort_values('observed_exposure')
axes[0].barh(top15['title'], top15['observed_exposure'], color='#EF5350')
axes[0].set_title('Top 15 Ocupações Mais Expostas', fontsize=11)
axes[0].set_xlabel('Índice de exposição observada')

# Distribuição geral
axes[1].hist(df_job['observed_exposure'], bins=30, color='#90CAF9', edgecolor='white')
axes[1].set_title('Distribuição da Exposição (todas as ocupações)', fontsize=11)
axes[1].set_xlabel('Índice de exposição observada')
axes[1].set_ylabel('Nº de ocupações')

plt.suptitle('Exposição das Ocupações ao Uso Real do Claude', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'job_exposure.png', dpi=150)
plt.show()

print(f"\nMediana de exposição: {df_job['observed_exposure'].median():.3f}")
print(f"% de ocupações com exposição > 0.1: {(df_job['observed_exposure'] > 0.1).mean():.1%}")

## 7. Exportar tabela-resumo

In [ ]:
metrics = ['usage_per_capita_index', 'usage_pct', 'automation_pct', 'augmentation_pct']

df_summary = (
    df[
        (df['geography'] == 'country') &
        (df['facet']     == 'country') &
        (df['variable'].isin(metrics)) &
        (df['geo_id'].isin(COMPARABLES))
    ]
    .pivot_table(index='geo_id', columns='variable', values='value')
    .reset_index()
    .assign(country=lambda x: x['geo_id'].map(LABELS))
    .set_index('country')
    .drop(columns='geo_id')
    .sort_values('usage_per_capita_index', ascending=False)
)

df_summary.to_csv(OUTPUT_DIR / 'summary_comparables.csv')
df_summary.style \
    .format({'usage_per_capita_index': '{:.2f}',
             'usage_pct':              '{:.2%}',
             'automation_pct':         '{:.2%}',
             'augmentation_pct':       '{:.2%}'}) \
    .background_gradient(cmap='Blues', subset=['usage_per_capita_index'])

## 8. Próximos Passos

- [ ] Comparar evoluções temporais: `release_2026_01_15` e `release_2026_03_24`
- [ ] Cruzar tarefas O\*NET com dados IBGE/PNAD para estimar exposição por ocupação no Brasil
- [ ] Explorar curvas de aprendizado da release `2026-03-24`
- [ ] Alimentar a apresentação `02_presentation.ipynb` com os outputs gerados